# LAB 5 - Simulated Annealing
**Heating CLN025: annealing, unfolding and hysteresis**


Authors:

- Prof. Marco A. Deriu (marco.deriu@polito.it)
- Eric A. Zizzi (eric.zizzi@polito.it)
- Marcello Miceli (marcello.miceli@polito.it)

# Table of Contents

1. Simulated annealing
2. Annealing in GROMACS
3. The runs: three ramps and a temperature ladder
4. Running them
5. Did the temperature follow the schedule?
6. Unfolding along the ramps: hysteresis
7. The temperature ladder: where does the hairpin melt?
8. How does the hairpin come apart?
9. Plain MD or annealing: which explores more?
10. Your deliverable

**Learning outcomes:**
- write a heating and cooling schedule with the GROMACS `annealing` options
- follow the unfolding of a protein with the observables of LAB 4
- explain why the unfolding temperature seen in a simulation depends on the heating rate
- tell apart what short simulations can measure (kinetics of unfolding) from what they cannot (the equilibrium melting curve)
- compare the unfolding pathway with the literature

# 1. Simulated annealing

**Annealing** is the slow cooling of a molten material: given time at each temperature, the atoms settle into a well-ordered crystal, the state of lowest free energy.

**Simulated annealing** borrows the idea to search for low-energy states of complicated systems (Kirkpatrick, Gelatt & Vecchi, *Science* 1983, [doi:10.1126/science.220.4598.671](https://doi.org/10.1126/science.220.4598.671)). For a molecule, heat gives it the energy to cross the barriers between conformations; slow cooling lets it settle into a low-energy one:

- at high temperature the system visits high-energy regions and crosses high barriers;
- as the temperature drops, low-energy states become more and more probable (Boltzmann);
- only with infinitely slow cooling is the lowest state guaranteed.

<img src="imgs/simulated_annealing2.png" width="800">

In practice the cooling is never slow enough, so annealing cannot guarantee the global minimum; repeating it and finding the same answer several times makes it likely. This lab uses the same machinery for a second purpose: **heating CLN025 to watch it unfold**, at different rates.

# 2. Annealing in GROMACS

GROMACS anneals by changing the **reference temperature of the thermostat** over time. The schedule is a list of time points and temperatures, joined by straight lines, one list per temperature-coupling group:

| Option | Values | Meaning |
|---|---|---|
| `annealing` | `no`, `single`, `periodic` (one per group) | `single`: run the schedule once, then stay at its last temperature. `periodic`: start over after the last point |
| `annealing-npoints` | integer (one per group) | Number of points of the schedule |
| `annealing-time` | ps (npoints values per group) | Time of each point |
| `annealing-temp` | K (npoints values per group) | Reference temperature at each point |

The course couples the whole system as one group (`tc-grps = System`), so each option takes a single value or list.

<div class="alert alert-block alert-warning">
<b>The thermostat follows, it does not jump.</b> Only the reference temperature changes along the schedule; the system relaxes towards it with the thermostat's time constant (<code>tau-t</code> = 0.1 ps in the course). A schedule that changes faster than the system can follow is not obeyed. After a run, always check that the temperature did what you asked (section 5).
</div>

An example: heat from 300 K to 500 K in 500 ps, stay at 500 K for 500 ps, cool back to 300 K in 500 ps:

```
annealing          = single
annealing-npoints  = 4
annealing-time     = 0   500  1000  1500
annealing-temp     = 300 500  500   300
```

`grompp` prints the schedule it understood: read it.

# 3. The runs

All runs start from the end of the 300 K run of LAB 3 (NMR model 1, AMBER), each with new random velocities, and run at **constant volume**. Above 373 K, the boiling point of water at 1 bar, a barostat would let the box expand until the water became a dilute gas and the simulation meaningless. At constant volume the water keeps the density of the liquid: it becomes a hot, compressed liquid under a pressure of thousands of bar. That is nothing like physiological conditions; it is simply a fast way to unfold the hairpin.

**Three ramps**, 300 → 500 → 300 K, at three speeds:

| Ramp | Heating | At 500 K | Cooling | Total | Heating rate |
|---|---|---|---|---|---|
| fast | 0.5 ns | 0.5 ns | 0.5 ns | 1.5 ns | 400 K/ns |
| medium | 2 ns | 1 ns | 2 ns | 5 ns | 100 K/ns |
| slow | 5 ns | 1 ns | 5 ns | 11 ns | 40 K/ns |

**A temperature ladder**: 5 ns at constant temperature, at 300, 350, 400, 450 and 500 K.

The course ran every run twice, with different velocities (two **replicas**). Download them in a terminal with `course-fetch --lab 05-SimulatedAnnealing --solutions`; they land in `solutions/`, whole (trajectory of the entire system, energies, log) and with the peptide alone already prepared for the analysis. You will run a few yourself (section 4).

## 3.1 The starting point

Choose where your LAB 3 run is, as in LAB 4 (`../03-ClassicalMD/solutions` for the course's):

In [ ]:
%env LAB3=../03-ClassicalMD

In [ ]:
%%bash
mkdir -p runs/ramp runs/ladder
cp $LAB3/cln025_model1/04-md/md_300K.gro runs/start.gro
cp $LAB3/cln025_model1/00-build/topol.top $LAB3/cln025_model1/00-build/posre.itp runs/
ls runs

## 3.2 The parameter files

Every run starts from the course's production file, `../common/mdp/md.mdp`, with the barostat off and new velocities. The ramps add the annealing schedule; the ladder only changes the temperature:

In [ ]:
import re

base = open("../common/mdp/md.mdp").read()
base = re.sub(r"(?m)^pcoupl .*", "pcoupl          = no        ; constant volume", base)
base = re.sub(r"(?m)^continuation .*", "continuation    = no", base)


def new_velocities(mdp, temperature):
    return re.sub(r"(?m)^gen-vel .*", f"gen-vel         = yes\ngen-temp        = {temperature}\ngen-seed        = -1", mdp)


ramps = {"fast": (500, 500), "medium": (2000, 1000), "slow": (5000, 1000)}   # heating (= cooling) and hold, in ps
for name, (heat, hold) in ramps.items():
    total = 2 * heat + hold
    mdp = re.sub(r"(?m)^nsteps .*", f"nsteps          = {total * 500}", new_velocities(base, 300))
    mdp += (f"\n; Simulated annealing: 300 K -> 500 K in {heat} ps, {hold} ps at 500 K, back to 300 K in {heat} ps\n"
            f"annealing               = single\nannealing-npoints       = 4\n"
            f"annealing-time          = 0 {heat} {heat + hold} {total}\nannealing-temp          = 300 500 500 300\n")
    with open(f"runs/ramp/ramp_{name}.mdp", "w") as f:
        f.write(mdp)

for T in (300, 350, 400, 450, 500):
    mdp = re.sub(r"(?m)^nsteps .*", "nsteps          = 2500000", new_velocities(base, T))
    mdp = re.sub(r"(?m)^ref-t .*", f"ref-t           = {T}", mdp)
    with open(f"runs/ladder/ladder_{T}.mdp", "w") as f:
        f.write(mdp)
print(open("runs/ramp/ramp_fast.mdp").read()[-400:])

In [ ]:
%%bash
cd runs
for mdp in ramp/*.mdp ladder/*.mdp; do
    gmx grompp -f $mdp -c start.gro -p topol.top -o ${mdp%.mdp}.tpr > ${mdp%.mdp}.grompp.out 2>&1 || tail -n 20 ${mdp%.mdp}.grompp.out
done
grep -A 3 "annealing" ramp/ramp_fast.grompp.out | head -n 8
ls ramp/*.tpr ladder/*.tpr

# 4. Running them

Together the runs are 42.5 ns (17.5 ns of ramps, 25 ns of ladder): too much for one person, easy for a class. Run the **fast ramp** and **one ladder temperature** (ask your instructor which; the ladder is shared out among the class), in the background from a terminal as in LAB 3:

```bash
cd ~/molbiomech/labs/05-SimulatedAnnealing/runs
nohup bash -c "cd ramp && gmx mdrun -deffnm ramp_fast -ntmpi 1 -v > ramp_fast.out 2>&1; \
               cd ../ladder && gmx mdrun -deffnm ladder_450 -ntmpi 1 -v > ladder_450.out 2>&1" > /dev/null 2>&1 &
```

Use your ns/day from LAB 3 to know how long they take (the fast ramp is 1.5 ns, a ladder run 5 ns). The analysis below works on your runs, on the course's, or on both.

## 4.1 Keep the peptide

As in LAB 4, the analysis uses only the peptide. This cell prepares every finished run in `runs/` (the course's runs in `solutions/` are already prepared):

In [ ]:
%%bash
cd runs
for xtc in ramp/*.xtc ladder/*.xtc; do
    [ -f "$xtc" ] || continue
    [[ $xtc == *_protein.xtc ]] && continue
    run=${xtc%.xtc}
    printf "Protein\nProtein\n" | gmx trjconv -s $run.tpr -f $xtc -o ${run}_protein.xtc -pbc mol -center > $run.prep.out 2>&1
    printf "Protein\nProtein\n" | gmx trjconv -s $run.tpr -f $xtc -o ${run}_protein.pdb -pbc mol -center -dump 0 >> $run.prep.out 2>&1
    echo Protein | gmx convert-tpr -s $run.tpr -o ${run}_protein.tpr >> $run.prep.out 2>&1
    echo "prepared $run"
done

Load every run found, yours in `runs/` and the course's in `solutions/`, and compute the course's observables (the definitions of LAB 4, in `../common/scripts/cln025.py`) together with the temperature, frame by frame:

In [ ]:
import sys
import glob
import os
import numpy as np
import matplotlib.pyplot as plt
import mdtraj as md

sys.path.append("../common/scripts")
import cln025
from cln025 import xvg


def load_run(prefix):
    """Observables and temperature of one run; prefix is e.g. runs/ramp/ramp_fast."""
    traj = md.load(prefix + "_protein.xtc", top=prefix + "_protein.pdb")
    obs = cln025.observables(traj)
    obs["time"] = traj.time                                  # ps
    out = prefix + ".temperature.xvg"
    if not os.path.exists(out):
        os.system(f"echo Temperature | gmx energy -f {prefix}.edr -o {out} > /dev/null 2>&1")
    energy = xvg(out)
    obs["T"] = np.interp(obs["time"], energy[:, 0], energy[:, 1])
    obs["traj"] = traj
    return obs


data = {}
for folder in ["runs", "solutions"]:
    for path in sorted(glob.glob(f"{folder}/*/*_protein.xtc")):
        prefix = path[:-len("_protein.xtc")]
        data[prefix] = load_run(prefix)
for prefix, obs in data.items():
    print(f"{prefix:<32} {obs['time'][-1] / 1000:5.1f} ns, folded {cln025.folded(obs).mean():.2f} of the frames")

# 5. Did the temperature follow the schedule?

In [ ]:
def schedule(name):
    heat, hold = ramps[name]
    return [0, heat, heat + hold, 2 * heat + hold], [300, 500, 500, 300]


fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, name in zip(axes, ramps):
    for prefix, obs in data.items():
        if os.path.basename(prefix).startswith(f"ramp_{name}"):
            ax.plot(obs["time"] / 1000, obs["T"], lw=0.6, label=prefix)
    t, T = schedule(name)
    ax.plot(np.array(t) / 1000, T, "k--", label="schedule")
    ax.set_title(f"{name} ramp")
    ax.set_xlabel("time (ns)")
axes[0].set_ylabel("temperature (K)")
axes[0].legend(fontsize=7)
fig.tight_layout()

# 6. Unfolding along the ramps: hysteresis

Plot the fraction of native contacts Q against the temperature of each frame, separating the heating from the cooling half of each ramp:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
print(f"{'run':<32} {'T when Q first < 0.5 (K)':>25} {'Q at the end':>13}")
for ax, name in zip(axes, ramps):
    heat, hold = ramps[name]
    for prefix, obs in data.items():
        if not os.path.basename(prefix).startswith(f"ramp_{name}"):
            continue
        heating = obs["time"] <= heat + hold
        ax.plot(obs["T"][heating], obs["q"][heating], ".", ms=2, color="tab:red")
        ax.plot(obs["T"][~heating], obs["q"][~heating], ".", ms=2, color="tab:blue")
        below = np.nonzero(obs["q"] < 0.5)[0]
        t_unfold = f"{obs['T'][below[0]]:.0f}" if len(below) else "never"
        print(f"{prefix:<32} {t_unfold:>25} {obs['q'][-20:].mean():13.2f}")
    ax.set_title(f"{name} ramp")
    ax.set_xlabel("temperature (K)")
axes[0].set_ylabel("Q")
axes[0].plot([], [], ".", color="tab:red", label="heating")
axes[0].plot([], [], ".", color="tab:blue", label="cooling")
axes[0].legend(markerscale=5)
fig.tight_layout()
fig.savefig("hysteresis.png", dpi=300)

<div class="alert alert-block alert-info">
<ul>
<li>At what temperature does the hairpin open on heating, and does it depend on the heating rate? Why would a faster ramp open it at a higher temperature?</li>
<li>Does the hairpin fold back on cooling? The heating and cooling branches do not coincide: this is <b>hysteresis</b>, the signature of a process slower than the ramp. Folding CLN025 takes hundreds of nanoseconds or more: even at the highest temperature measured, its relaxation times are about 100 ns (Davis et al., <i>J Am Chem Soc</i> 2012, <a href="https://doi.org/10.1021/ja3046734">doi:10.1021/ja3046734</a>). How slow would the cooling need to be?</li>
<li>So, as a search method, can annealing find the folded hairpin from the unfolded one in these runs?</li>
</ul>
</div>

# 7. The temperature ladder: where does the hairpin melt?

Experimentally, CLN025 unfolds reversibly, like a two-state system, with a melting temperature of **70 °C (343 K)** (Honda, *Seibutsu Butsuri* 2009, [doi:10.2142/biophys.49.126](https://doi.org/10.2142/biophys.49.126); Honda et al., *J Am Chem Soc* 2008, [doi:10.1021/ja8030533](https://doi.org/10.1021/ja8030533)). That value is for CLN025 without caps, the molecule of the experiment.

Follow the folded fraction (Q ≥ 0.7 and RMSD < 0.15 nm, as in LAB 4) in each ladder run:

In [ ]:
temperatures = [300, 350, 400, 450, 500]
fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.5))
summary = {T: [] for T in temperatures}
for prefix, obs in data.items():
    base_name = os.path.basename(prefix)
    if not base_name.startswith("ladder_"):
        continue
    T = int(base_name.split("_")[1])
    fold = cln025.folded(obs)
    summary[T].append(fold.mean())
    window = min(25, len(fold))                              # running average over at most 250 ps
    running = np.convolve(fold, np.ones(window) / window, mode="valid")
    left.plot(obs["time"][window - 1:] / 1000, running, lw=0.8, color=plt.cm.coolwarm((T - 300) / 200))
for T in temperatures:
    if summary[T]:
        right.plot([T] * len(summary[T]), summary[T], "o", color=plt.cm.coolwarm((T - 300) / 200))
right.plot([T for T in temperatures if summary[T]], [np.mean(summary[T]) for T in temperatures if summary[T]], "k-")
right.axvline(343, color="gray", ls="--", label="experimental $T_m$ (uncapped)")
left.set_xlabel("time (ns)")
left.set_ylabel("folded fraction (running average)")
right.set_xlabel("temperature (K)")
right.set_ylabel("folded fraction over the run")
right.legend()
fig.tight_layout()
fig.savefig("ladder.png", dpi=300)
for T in temperatures:
    print(f"{T} K: folded fraction " + ", ".join(f"{f:.2f}" for f in summary[T]))

<div class="alert alert-block alert-info">
<ul>
<li>Every run starts folded. At 350 K, just above the experimental melting temperature, does the hairpin unfold within 5 ns? What does that tell you about AMBER's stabilisation of the fold (LAB 4), and about the time scale?</li>
<li>Is the curve on the right a <b>melting curve</b>? A melting curve is an equilibrium property: at each temperature the system must unfold <i>and refold</i> many times during the run. How long would each run need to be, given relaxation times of 100 ns or more? What you measured instead is how fast the hairpin unfolds at each temperature: kinetics, not thermodynamics.</li>
<li>For the same reason, these runs cannot give the unfolding enthalpy ΔH by a van 't Hoff fit. For comparison, a simulation that did sample folding and unfolding many times, with an enhanced-sampling method, found ΔH = 24.1 ± 4.9 kJ/mol for chignolin (Okumura, <i>Proteins</i> 2012, <a href="https://doi.org/10.1002/prot.24125">doi:10.1002/prot.24125</a>).</li>
</ul>
</div>

# 8. How does the hairpin come apart?

Okumura (2012) found that on heating, chignolin opens from the C-terminus, or from both ends at once, and may pass through an α-helix or 3₁₀-helix before unfolding completely. Check it in the hottest runs: how much of the time is each of the three backbone hydrogen bonds of LAB 4 formed, when does it first break for at least 200 ps, and is there any helix?

In [ ]:
bonds = {"Asp3 N - Thr8 O": ((3, "N"), (8, "O")), "Gly7 N - Asp3 O": ((7, "N"), (3, "O")),
         "Thr8 N - Asp3 O": ((8, "N"), (3, "O")), "Tyr1 CA - Tyr10 CA": ((1, "CA"), (10, "CA"))}
cutoffs = {"Tyr1 CA - Tyr10 CA": 0.8}                          # open end-to-end: above 0.8 nm


def first_long_break(time, distance, cutoff, least_ps=200):
    """First time the distance stays above the cutoff for at least least_ps (on a 100 ps running average)."""
    smooth = np.convolve(distance, np.ones(10) / 10, mode="same")
    needed = max(1, round(least_ps / (time[1] - time[0])))
    broken = 0
    for i, is_broken in enumerate(smooth >= cutoff):
        broken = broken + 1 if is_broken else 0
        if broken == needed:
            return time[i - needed + 1]
    return None


print(f"{'run':<32}" + "".join(f"{b:>22}" for b in bonds) + f"{'helix':>7}")
print(f"{'':<32}" + "".join(f"{'formed   first break':>22}" for b in bonds))
for prefix, obs in data.items():
    base_name = os.path.basename(prefix)
    if not (base_name.startswith("ladder_450") or base_name.startswith("ladder_500")):
        continue
    traj = obs["traj"]
    index = {(a.residue.resSeq, a.name): a.index for a in traj.topology.atoms}
    row = f"{prefix:<32}"
    for bond, (a, b) in bonds.items():
        cutoff = cutoffs.get(bond, 0.35)
        d = md.compute_distances(traj, [(index[a], index[b])])[:, 0]
        t = first_long_break(obs["time"], d, cutoff)
        row += f"{100 * (d < cutoff).mean():9.0f}% {'never' if t is None else f'{t / 1000:.1f} ns':>11}"
    ss = md.compute_dssp(traj, simplified=False)[:, 1:-1]
    helix = np.isin(ss, ["H", "G", "I"]).any(axis=1).mean()
    print(row + f"{helix:7.2f}")

A bond counts as broken only if it stays broken for 200 ps: at these temperatures every bond flickers, and a single frame above the cut-off means nothing. Open a 500 K run in VMD to watch it: from a terminal in `runs/ladder` (or `solutions/ladder`),

```bash
vmd ladder_500_protein.pdb ladder_500_protein.xtc
```

with NewCartoon coloured by Structure, and Tyr1, Tyr2, Trp9 and Tyr10 as Licorice.

<div class="alert alert-block alert-info">Which bond is formed least of the time, and which breaks first? Does the C-terminal end (Thr8, Trp9, Tyr10) come apart before the N-terminal one, as Okumura found? Does the end-to-end distance ever open, and how much helix do you see? Remember that Okumura simulated chignolin without caps, with a different AMBER force field, and for far longer than 5 ns.</div>

# 9. Plain MD or annealing: which explores more?

Clustering (LAB 4) counts how many distinct structures a run visits. Compare the 300 K run of LAB 3 with the slow ramp, with the same method and cut-off:

In [ ]:
%%bash
mkdir -p clusters
src=$LAB3/cln025_model1/04-md/md_300K
printf "Protein\nProtein\n" | gmx trjconv -s $src.tpr -f $src.xtc -o clusters/md_300K_protein.xtc -pbc mol -center > clusters/prep.out 2>&1
echo Protein | gmx convert-tpr -s $src.tpr -o clusters/md_300K_protein.tpr >> clusters/prep.out 2>&1
slow=$(ls runs/ramp/ramp_slow_protein.xtc solutions/ramp/ramp_slow_1_protein.xtc 2>/dev/null | head -n 1)
for run in clusters/md_300K_protein ${slow%.xtc}; do
    printf "Backbone\nProtein\n" | gmx cluster -s $run.tpr -f $run.xtc -dt 20 -method gromos -cutoff 0.1 \
        -g clusters/$(basename $run).log -cl clusters/$(basename $run).pdb > clusters/cluster.out 2>&1 || tail -n 20 clusters/cluster.out
    echo "$(basename $run): $(grep 'Found' clusters/$(basename $run).log)"
done

<div class="alert alert-block alert-info">Why does the annealing run visit so many more structures? Most of them are unfolded: is "exploring more" the same as "finding the right structure"? Open the cluster middles (<code>clusters/*.pdb</code>) in VMD.</div>

# 10. Your deliverable

1. **The temperature check** of section 5: did your runs follow their schedules?
2. **The hysteresis plot** of section 6, with the opening temperature of each ramp and an explanation of why it depends on the rate.
3. **The ladder** of section 7, with a paragraph on what it measures, what a real melting curve would need, and how it compares with the experimental 343 K.
4. **The unfolding pathway** of section 8, compared with Okumura (2012), with a VMD figure of the hairpin coming apart.

As for the other simulation labs, present them in a few slides during a 15-minute group discussion.